In [ ]:
import sys
import warnings

import matplotlib
import matplotlib.pyplot as plt
import time
import torch

sys.path = [p for p in sys.path if "dist-packages" not in p or ".venv" in p]
warnings.filterwarnings("ignore")

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)

print(f"Device:            {torch.cuda.get_device_name(0)}")
print(f"Compute Capability:{props.major}.{props.minor}")
print(f"SMs:               {props.multi_processor_count}")
print(f"Total Memory:      {props.total_memory / 1024**3:.1f} GB")
print(f"PyTorch:           {torch.__version__}")
print(f"CUDA:              {torch.version.cuda}")
print(f"Matplotlib:        {matplotlib.__version__}")

In [ ]:
def benchmark_matmul(M, K, N, dtype, num_warmup=10, num_iters=100, label=None):
    """Benchmark matrix multiplication: (M x K) @ (K x N)"""
    if dtype == torch.int8:
        a = torch.randint(-128, 127, (M, K), dtype=torch.int8, device=device)
        b = torch.randint(-128, 127, (K, N), dtype=torch.int8, device=device)
    else:
        a = torch.randn(M, K, device=device, dtype=dtype)
        b = torch.randn(K, N, device=device, dtype=dtype)
    # warmup
    for _ in range(num_warmup):
        if dtype == torch.int8:
            _ = torch._int_mm(a, b)
        else:
            _ = torch.mm(a, b)
    torch.cuda.synchronize()

    # timed run
    start = time.perf_counter()
    for _ in range(num_iters):
        if dtype == torch.int8:
            _ = torch._int_mm(a, b)
        else:
            _ = torch.mm(a, b)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    ops = 2 * M * K * N * num_iters  # multiply-add = 2 ops
    tflops = ops / elapsed / 1e12
    avg_ms = elapsed / num_iters * 1000

    label = label or str(dtype).split(".")[-1]
    print(f"  {label:>10s}  {M}x{K} @ {K}x{N}  {avg_ms:7.2f} ms  {tflops:.3f} TFLOPS")
    return {"dtype": label, "M": M, "K": K, "N": N, "ms": avg_ms, "tflops": tflops}

In [4]:
print("=" * 70)
print("DTYPE COMPARISON — 4096x4096 matmul")
print("=" * 70)

results = []
size = 4096

# FP32 without TF32 (pure CUDA cores)
torch.backends.cuda.matmul.allow_tf32 = False
results.append(benchmark_matmul(size, size, size, torch.float32, label="fp32-pure"))

# FP32 with TF32 (Tensor Cores, default PyTorch behavior)
torch.backends.cuda.matmul.allow_tf32 = True
results.append(benchmark_matmul(size, size, size, torch.float32, label="tf32"))

# FP16
results.append(benchmark_matmul(size, size, size, torch.float16, label="fp16"))

# BF16
results.append(benchmark_matmul(size, size, size, torch.bfloat16, label="bf16"))

# INT8
results.append(benchmark_matmul(size, size, size, torch.int8, label="int8"))

# reset to default
torch.backends.cuda.matmul.allow_tf32 = True

DTYPE COMPARISON — 4096x4096 matmul
   fp32-pure  4096x4096 @ 4096x4096   166.09 ms  0.827 TFLOPS
        tf32  4096x4096 @ 4096x4096    35.50 ms  3.872 TFLOPS
        fp16  4096x4096 @ 4096x4096    22.92 ms  5.996 TFLOPS
        bf16  4096x4096 @ 4096x4096    16.57 ms  8.296 TFLOPS
        int8  4096x4096 @ 4096x4096    59.57 ms  2.307 TFLOPS


In [5]:
print("=" * 70)
print("SIZE SCALING - FP16 matmul across sizes")
print("=" * 70)

size_results = []
for size in [128, 256, 512, 1024, 2048, 4096, 8192]:
    try:
        r = benchmark_matmul(size, size, size, torch.float16, label=f"fp16-{size}")
        size_results.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"  {'fp16-'+str(size):>10s}  OOM")
        break
    torch.cuda.empty_cache()

SIZE SCALING - FP16 matmul across sizes
    fp16-128  128x128 @ 128x128     0.37 ms  0.011 TFLOPS
    fp16-256  256x256 @ 256x256     0.13 ms  0.261 TFLOPS
    fp16-512  512x512 @ 512x512     0.13 ms  2.102 TFLOPS
   fp16-1024  1024x1024 @ 1024x1024     0.30 ms  7.077 TFLOPS
   fp16-2048  2048x2048 @ 2048x2048     2.49 ms  6.907 TFLOPS
   fp16-4096  4096x4096 @ 4096x4096    23.30 ms  5.900 TFLOPS
   fp16-8192  8192x8192 @ 8192x8192   169.16 ms  6.500 TFLOPS


In [6]:
print("=" * 70)
print("MEMORY BANDWIDTH")
print("=" * 70)

for dtype, name in [(torch.float32, "fp32"), (torch.float16, "fp16")]:
    nbytes = dtype.itemsize
    n = 50_000_000  # 50M elements
    a = torch.randn(n, device=device, dtype=dtype)
    b = torch.empty_like(a)

    # warmup
    for _ in range(5):
        b.copy_(a)
    torch.cuda.synchronize()

    iters = 100
    start = time.perf_counter()
    for _ in range(iters):
        b.copy_(a)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    # read + write
    total_bytes = 2 * n * nbytes * iters
    gbps = total_bytes / elapsed / 1e9
    print(f"  {name}: {gbps:.1f} GB/s  ({n * nbytes / 1e6:.0f} MB per transfer)")

del a, b
torch.cuda.empty_cache()

MEMORY BANDWIDTH
  fp32: 61.8 GB/s  (200 MB per transfer)
  fp16: 62.6 GB/s  (100 MB per transfer)


In [7]:
print("=" * 70)
print("SUSTAINED COMPUTE — GPU memory usage across dtypes")
print("=" * 70)

for dtype, name in [
    (torch.float32, "fp32"),
    (torch.float16, "fp16"),
    (torch.bfloat16, "bf16"),
]:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    size = 4096
    a = torch.randn(size, size, device=device, dtype=dtype)
    b = torch.randn(size, size, device=device, dtype=dtype)
    c = torch.mm(a, b)
    torch.cuda.synchronize()

    alloc = torch.cuda.memory_allocated() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2
    print(f"  {name}: allocated={alloc:.0f} MB, peak={peak:.0f} MB  "
          f"(matrix footprint: {3 * size**2 * dtype.itemsize / 1024**2:.0f} MB)")

    del a, b, c
    torch.cuda.empty_cache()

SUSTAINED COMPUTE — GPU memory usage across dtypes
  fp32: allocated=200 MB, peak=200 MB  (matrix footprint: 192 MB)
  fp16: allocated=104 MB, peak=104 MB  (matrix footprint: 96 MB)
  bf16: allocated=104 MB, peak=104 MB  (matrix footprint: 96 MB)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. TFLOPS by dtype
dtypes = ["fp32-pure", "tf32", "fp16", "bf16", "int8"]
tflops = [0.825, 3.862, 6.029, 8.279, 2.307]
colors = ["#888", "#4a9", "#27a", "#e54", "#fa0"]
axes[0].barh(dtypes, tflops, color=colors)
axes[0].set_xlabel("TFLOPS")
axes[0].set_title("4096x4096 Matmul Throughput")
for i, v in enumerate(tflops):
    axes[0].text(v + 0.1, i, f"{v:.2f}", va="center", fontsize=9)

# 2. TFLOPS vs matrix size (FP16)
sizes = [128, 256, 512, 1024, 2048, 4096, 8192]
tflops_sz = [0.011, 0.277, 2.268, 7.099, 6.929, 5.902, 6.463]
axes[1].plot(sizes, tflops_sz, "o-", color="#27a", linewidth=2)
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("Matrix dimension (N)")
axes[1].set_ylabel("TFLOPS")
axes[1].set_title("FP16 Scaling by Size")
axes[1].axhline(y=7.1, color="#ccc", linestyle="--", linewidth=1)

# 3. Memory footprint
mem_dtypes = ["fp32", "fp16", "bf16"]
mem_mb = [200, 104, 104]
axes[2].bar(mem_dtypes, mem_mb, color=["#888", "#27a", "#e54"])
axes[2].set_ylabel("MB")
axes[2].set_title("3x 4096x4096 Memory Usage")
for i, v in enumerate(mem_mb):
    axes[2].text(i, v + 3, f"{v} MB", ha="center", fontsize=9)

plt.suptitle("Jetson Orin Nano — Compute Characterization", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("orin-nano-benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey findings:")
print("  Peak throughput:    8.3 TFLOPS (BF16)")
print("  Memory bandwidth:  62 GB/s (shared CPU/GPU)")
print("  Sweet spot size:   1024+ (compute-bound)")
print("  Best dtype for CO: BF16 (10x over FP32, same memory as FP16)")